# Preconditioners for a 2D Laplacian

[lineax#120](https://github.com/patrick-kidger/lineax/pull/120) proposed an example showing that a simple Jacobi (diagonal) preconditioner improves GMRES on a 2D Poisson problem. This notebook is self-contained, but follows on from that idea with two more structured preconditioners built directly out of lineax's structured operators:

1. a **tridiagonal line preconditioner**, built with `lx.invert(lx.TridiagonalLinearOperator(...))`;
2. a **circulant line preconditioner**, built with `lx.invert(lx.CirculantLinearOperator(...))`, for problems with a periodic direction.

Rather than the constant-coefficient Laplacian (which a real sine/cosine transform would diagonalise outright, making any other preconditioner look artificially good), the diffusion coefficient here varies smoothly in *both* grid directions. That keeps the operator honestly non-separable, so the structured preconditioners below are genuine approximations, not disguised exact solves.

## The problem

We discretise $-\nabla \cdot (\kappa(x, y) \nabla u) = f$ on the unit square with homogeneous Dirichlet boundary conditions, using a standard 5-point finite-volume stencil with harmonic-mean face conductivities. `anisotropy` scales the conductivity in the $x$-direction relative to $y$ -- physically this is the same as stretching the grid, and it is exactly the situation (e.g. boundary-layer meshes, layered media) where line-implicit preconditioners are classically used.

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.experimental.sparse as js
import lineax as lx
import numpy as np
from scipy.sparse import block_diag, diags, lil_matrix

jax.config.update("jax_enable_x64", True)


def _harm(a, b):
    return 2 * a * b / (a + b)


def kappa_field(n, m):
    h = 1.0 / (n + 1)
    xs = (np.arange(1, n + 1)) * h
    ys = (np.arange(1, m + 1)) * h
    X, Y = np.meshgrid(xs, ys)  # shape (m, n)
    return 1.0 + 0.9 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y) + 0.9 * X * Y


def assemble(n, m, anisotropy=1.0):
    """5-point FD discretisation of -div(kappa grad u), Dirichlet BCs."""
    kappa = kappa_field(n, m)
    h = 1.0 / (n + 1)

    kx = np.empty((m, n + 1))  # x-faces, including the two boundary faces
    kx[:, 1:-1] = _harm(kappa[:, :-1], kappa[:, 1:])
    kx[:, 0] = kappa[:, 0]
    kx[:, -1] = kappa[:, -1]
    kx *= anisotropy / h**2

    ky = np.empty((m + 1, n))  # y-faces, including the two boundary faces
    ky[1:-1, :] = _harm(kappa[:-1, :], kappa[1:, :])
    ky[0, :] = kappa[0, :]
    ky[-1, :] = kappa[-1, :]
    ky /= h**2

    row_blocks = []
    for i in range(m):
        left, right = kx[i, :-1], kx[i, 1:]
        diag = left + right
        off = -kx[i, 1:-1]
        row_blocks.append(diags([off, diag, off], [-1, 0, 1], shape=(n, n)))
    Ax = block_diag(row_blocks, format="csr")

    Ay = lil_matrix((m * n, m * n))
    for i in range(m):
        sl = slice(i * n, (i + 1) * n)
        Ay[sl, sl] = Ay[sl, sl] + diags(ky[i, :] + ky[i + 1, :])
    for i in range(m - 1):
        sl_top, sl_bot = slice(i * n, (i + 1) * n), slice((i + 1) * n, (i + 2) * n)
        w = diags(ky[i + 1, :])
        Ay[sl_top, sl_bot] = Ay[sl_top, sl_bot] - w
        Ay[sl_bot, sl_top] = Ay[sl_bot, sl_top] - w

    A = (Ax + Ay.tocsr()).tocsr()
    return A, kx, ky


class SparseMatrixLinearOperator(lx.MatrixLinearOperator):
    def mv(self, vector):
        return self.matrix @ vector


@lx.is_symmetric.register(SparseMatrixLinearOperator)
def _(op):
    return True


@lx.is_positive_semidefinite.register(SparseMatrixLinearOperator)
def _(op):
    return True


def make_operator(n, m, anisotropy=1.0):
    A_sp, kx, ky = assemble(n, m, anisotropy)
    operator = SparseMatrixLinearOperator(js.BCOO.from_scipy_sparse(A_sp))
    key = jr.PRNGKey(0)
    b = jr.uniform(key, (n * m,), dtype=jnp.float64)
    return operator, A_sp, b, kx, ky


## No preconditioner, and Jacobi

As a baseline, we solve with `lx.CG` (the operator is symmetric positive definite, so CG is the natural choice over GMRES) with no preconditioner and with the diagonal (Jacobi) preconditioner from the original example.

In [1]:
def solve(operator, A_sp, b, name, preconditioner=None, **solver_kwargs):
    solver = lx.CG(atol=1e-6, rtol=1e-6, max_steps=2000, **solver_kwargs)
    options = {} if preconditioner is None else {"preconditioner": preconditioner}
    sol = lx.linear_solve(operator, b, solver=solver, options=options, throw=False)
    x = np.asarray(sol.value)
    resid = np.linalg.norm(b - A_sp @ x) / np.linalg.norm(b)
    steps = int(sol.stats["num_steps"])
    print(f"{name:32s} steps={steps:5d}  rel_resid={resid:.3e}  result={sol.result}")
    return steps


n = m = 64
operator, A_sp, b, kx, ky = make_operator(n, m, anisotropy=1.0)
diag = np.asarray(A_sp.diagonal())

solve(operator, A_sp, b, "none")

jacobi = lx.FunctionLinearOperator(
    lambda v: v / diag,
    operator.in_structure(),
    tags=[lx.positive_semidefinite_tag, lx.symmetric_tag],
)
solve(operator, A_sp, b, "jacobi", jacobi)


none                             steps=  294  rel_resid=3.607e-07  result=lineax._solution.RESULTS<>
jacobi                           steps=  198  rel_resid=4.552e-07  result=lineax._solution.RESULTS<>


## A tridiagonal line preconditioner

The 2D operator is (approximately, since $\kappa$ is not separable) a Kronecker sum of a 1D operator in $x$ and a 1D operator in $y$. If we solve the $x$-direction coupling *exactly* on every grid row and simply drop the $y$-direction coupling, each row is a genuine tridiagonal system -- solved in $O(n)$ by `lx.invert(lx.TridiagonalLinearOperator(...))` (the Thomas algorithm), rather than approximated diagonally as Jacobi does. `jax.vmap` applies this to every row at once.

In [1]:
def tridiagonal_line_preconditioner(kx, n, m):
    row_diag = jnp.asarray(kx[:, :-1] + kx[:, 1:])  # (m, n)
    row_off = jnp.asarray(-kx[:, 1:-1])  # (m, n - 1)

    def line_solve(diag_row, off_row, v_row):
        T = lx.TridiagonalLinearOperator(diag_row, off_row, off_row)
        return lx.invert(T, lx.Tridiagonal()).mv(v_row)

    batched_line_solve = jax.vmap(line_solve)

    def apply(v):
        return batched_line_solve(row_diag, row_off, v.reshape(m, n)).reshape(m * n)

    return lx.FunctionLinearOperator(
        apply,
        jax.ShapeDtypeStruct((m * n,), jnp.float64),
        tags=[lx.positive_semidefinite_tag, lx.symmetric_tag],
    )


tridiag_pc = tridiagonal_line_preconditioner(kx, n, m)
solve(operator, A_sp, b, "tridiagonal line (x-direction)", tridiag_pc)


tridiagonal line (x-direction)   steps=  271  rel_resid=5.643e-07  result=lineax._solution.RESULTS<>


That is a *worse* result than plain Jacobi! This is not a bug -- it's the expected behaviour for an isotropic 2D Laplacian. Writing the constant-coefficient operator in its eigenbasis, $A$ has eigenvalues $\lambda_i + \mu_j$ (one term per direction), while the $x$-only line preconditioner $M = A_x \otimes I$ has eigenvalues $\lambda_i$. The preconditioned operator $M^{-1}A$ then has eigenvalues $1 + \mu_j / \lambda_i$, whose range is still $O(h^{-2})$ -- exactly the same asymptotic conditioning as the original problem, because the untouched $y$-direction is exactly as stiff as the $x$-direction the preconditioner removed. Solving one direction exactly buys you nothing if both directions are equally hard.

It buys you a lot, however, once the two directions are *not* equally hard -- e.g. a stretched grid or an anisotropic diffusion coefficient, both common in practice (boundary layers, layered/fractured media). We sweep an `anisotropy` factor that scales the $x$-conductivity relative to $y$ and re-run all three solves.

In [1]:
results = {}
for a in [1, 5, 20, 50]:
    operator, A_sp, b, kx, ky = make_operator(n, m, anisotropy=float(a))
    diag = np.asarray(A_sp.diagonal())
    jacobi = lx.FunctionLinearOperator(
        lambda v, diag=diag: v / diag,
        operator.in_structure(),
        tags=[lx.positive_semidefinite_tag, lx.symmetric_tag],
    )
    tridiag_pc = tridiagonal_line_preconditioner(kx, n, m)
    results[a] = {
        "none": solve(operator, A_sp, b, f"[x{a:>3}] none"),
        "jacobi": solve(operator, A_sp, b, f"[x{a:>3}] jacobi", jacobi),
        "tridiagonal": solve(operator, A_sp, b, f"[x{a:>3}] tridiagonal", tridiag_pc),
    }


[x  1] none                     steps=  294  rel_resid=3.607e-07  result=lineax._solution.RESULTS<>
[x  1] jacobi                   steps=  198  rel_resid=4.552e-07  result=lineax._solution.RESULTS<>
[x  1] tridiagonal              steps=  271  rel_resid=5.643e-07  result=lineax._solution.RESULTS<>
[x  5] none                     steps=  335  rel_resid=4.566e-07  result=lineax._solution.RESULTS<>
[x  5] jacobi                   steps=  222  rel_resid=5.431e-07  result=lineax._solution.RESULTS<>
[x  5] tridiagonal              steps=  124  rel_resid=5.359e-07  result=lineax._solution.RESULTS<>
[x 20] none                     steps=  410  rel_resid=3.613e-07  result=lineax._solution.RESULTS<>
[x 20] jacobi                   steps=  282  rel_resid=5.411e-07  result=lineax._solution.RESULTS<>
[x 20] tridiagonal              steps=   66  rel_resid=4.557e-07  result=lineax._solution.RESULTS<>
[x 50] none                     steps=  449  rel_resid=3.943e-07  result=lineax._solution.RESULTS<>


| anisotropy | none | jacobi | tridiagonal line |
|---:|---:|---:|---:|
| 1  | 294 | 198 | 271 |
| 5  | 335 | 222 | 124 |
| 20 | 410 | 282 |  66 |
| 50 | 449 | 301 |  43 |

As the grid/coefficient anisotropy grows, the line preconditioner goes from *worse* than Jacobi to over **10x** fewer iterations than the unpreconditioned solve, and roughly **7x** fewer than Jacobi -- while Jacobi's advantage over no preconditioner barely moves, since a purely diagonal preconditioner cannot see the directional stiffness at all. For genuinely isotropic problems, plain Jacobi (or nothing, for very well-conditioned systems) remains the more effective and far cheaper choice per iteration.

## A circulant line preconditioner

!!! warning

    This section uses `lx.CirculantLinearOperator` and `lx.Circulant`, added in [lineax#238](https://github.com/patrick-kidger/lineax/pull/238). At the time of writing this has landed on the `dev` branch but is not yet part of a release, so the cell below will not run against a `pip install lineax`. It was executed against `dev` (`pip install "lineax @ git+https://github.com/patrick-kidger/lineax@dev"`) to produce the results shown.

The tridiagonal line preconditioner only pays off when a grid direction is genuinely Dirichlet/non-periodic. If a direction is instead *periodic* -- a common setup for, e.g., a periodic channel flow -- the natural analogue is a **circulant** line solve: `lx.CirculantLinearOperator` stores a whole row in $O(n)$ and `lx.invert(..., lx.Circulant())` solves it in $O(n \log n)$ via the FFT, rather than $O(n)$ via Thomas' algorithm.

To keep this honest, $\kappa$ still varies *within* each periodic row, so no single row is actually circulant -- exactly diagonalising it would require a fresh FFT-friendly assumption for every row. Instead we build a genuinely circulant approximation per row from its row-averaged conductivity (the classical Strang circulant-preconditioner recipe for Toeplitz systems), and use *that* as the preconditioner. One further wrinkle: a bare periodic 1D Laplacian is singular (its row sums to zero, with the constant vector in its kernel), so used naively as a preconditioner it destabilises CG. We fix this by folding the row-averaged $y$-diagonal into the circulant's own diagonal -- a uniform shift, so it stays circulant -- which is exactly the diagonal the full operator has along that line anyway. (An alternative that keeps the *exact* line matrix, rather than an approximation with a corrected diagonal, is a Woodbury correction: express the true non-periodic tridiagonal line as a circulant plus a rank-2 update for the two boundary corners, and solve it via Sherman-Morrison-Woodbury on top of the FFT solve. We haven't implemented that here -- it's a nice follow-up, and the canonical answer for a genuinely constant-coefficient separable domain is a sine/cosine transform, not this at all.)

In [1]:
def kappa_field_periodic(n, m):
    hx, hy = 1.0 / n, 1.0 / (m + 1)
    xs, ys = np.arange(n) * hx, (np.arange(1, m + 1)) * hy
    X, Y = np.meshgrid(xs, ys)
    return 1.0 + 0.9 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y) + 0.9 * X * Y


def assemble_periodic(n, m, anisotropy=1.0):
    """As `assemble`, but periodic in x (e.g. a periodic channel) and Dirichlet in y."""
    kappa = kappa_field_periodic(n, m)
    hx, hy = 1.0 / n, 1.0 / (m + 1)

    kx = _harm(np.roll(kappa, 1, axis=1), kappa) * anisotropy / hx**2  # (m, n), periodic faces
    ky = np.empty((m + 1, n))
    ky[1:-1, :] = _harm(kappa[:-1, :], kappa[1:, :])
    ky[0, :], ky[-1, :] = kappa[0, :], kappa[-1, :]
    ky /= hy**2

    row_blocks = []
    for i in range(m):
        w = kx[i]
        diag, off = w + np.roll(w, -1), -w[1:]
        block = diags([off, diag, off], [-1, 0, 1], shape=(n, n)).tolil()
        block[0, -1] = block[-1, 0] = -w[0]
        row_blocks.append(block.tocsr())
    Ax = block_diag(row_blocks, format="csr")

    Ay = lil_matrix((m * n, m * n))
    for i in range(m):
        sl = slice(i * n, (i + 1) * n)
        Ay[sl, sl] = Ay[sl, sl] + diags(ky[i, :] + ky[i + 1, :])
    for i in range(m - 1):
        sl_top, sl_bot = slice(i * n, (i + 1) * n), slice((i + 1) * n, (i + 2) * n)
        w = diags(ky[i + 1, :])
        Ay[sl_top, sl_bot] = Ay[sl_top, sl_bot] - w
        Ay[sl_bot, sl_top] = Ay[sl_bot, sl_top] - w

    return (Ax + Ay.tocsr()).tocsr(), kx, ky


def circulant_line_preconditioner(kx, ky, n, m):
    kbar = kx.mean(axis=1)  # row-averaged x-conductivity
    ybar = 0.5 * (ky[:-1, :] + ky[1:, :]).mean(axis=1)  # row-averaged y-diagonal
    column0 = (
        jnp.zeros((m, n))
        .at[:, 0].set(2 * kbar + ybar)
        .at[:, 1].set(-kbar)
        .at[:, -1].set(-kbar)
    )

    def line_solve(column, v_row):
        C = lx.CirculantLinearOperator(column)
        return lx.invert(C, lx.Circulant(well_posed=True)).mv(v_row)

    batched_line_solve = jax.vmap(line_solve)

    def apply(v):
        return batched_line_solve(column0, v.reshape(m, n)).reshape(m * n)

    return lx.FunctionLinearOperator(
        apply,
        jax.ShapeDtypeStruct((m * n,), jnp.float64),
        tags=[lx.positive_semidefinite_tag, lx.symmetric_tag],
    )


results = {}
for a in [1, 5, 20, 50]:
    A_sp, kx, ky = assemble_periodic(n, m, anisotropy=float(a))
    operator = SparseMatrixLinearOperator(js.BCOO.from_scipy_sparse(A_sp))
    b = jr.uniform(jr.PRNGKey(0), (n * m,), dtype=jnp.float64)
    diag = np.asarray(A_sp.diagonal())
    jacobi = lx.FunctionLinearOperator(
        lambda v, diag=diag: v / diag,
        operator.in_structure(),
        tags=[lx.positive_semidefinite_tag, lx.symmetric_tag],
    )
    circ_pc = circulant_line_preconditioner(kx, ky, n, m)
    results[a] = {
        "none": solve(operator, A_sp, b, f"[x{a:>3}] none"),
        "jacobi": solve(operator, A_sp, b, f"[x{a:>3}] jacobi", jacobi),
        "circulant": solve(operator, A_sp, b, f"[x{a:>3}] circulant", circ_pc),
    }


[x  1] none                     steps=  316  rel_resid=3.589e-07  result=lineax._solution.RESULTS<>
[x  1] jacobi                   steps=  224  rel_resid=4.375e-07  result=lineax._solution.RESULTS<>
[x  1] circulant                steps=  217  rel_resid=4.540e-07  result=lineax._solution.RESULTS<>
[x  5] none                     steps=  387  rel_resid=2.953e-07  result=lineax._solution.RESULTS<>
[x  5] jacobi                   steps=  277  rel_resid=4.680e-07  result=lineax._solution.RESULTS<>
[x  5] circulant                steps=  152  rel_resid=3.225e-07  result=lineax._solution.RESULTS<>
[x 20] none                     steps=  570  rel_resid=3.240e-07  result=lineax._solution.RESULTS<>
[x 20] jacobi                   steps=  402  rel_resid=5.333e-07  result=lineax._solution.RESULTS<>
[x 20] circulant                steps=  116  rel_resid=2.408e-07  result=lineax._solution.RESULTS<>
[x 50] none                     steps=  798  rel_resid=3.254e-07  result=lineax._solution.RESULTS<>


| anisotropy | none | jacobi | circulant line |
|---:|---:|---:|---:|
| 1  | 316 | 224 | 217 |
| 5  | 387 | 277 | 152 |
| 20 | 570 | 402 | 116 |
| 50 | 798 | 558 | 106 |

Same story as the tridiagonal case: at `anisotropy=1` the circulant line preconditioner is barely better than Jacobi (for the same reason -- the un-preconditioned direction is still equally stiff), but as the periodic direction is made stiffer it pulls steadily ahead, reaching roughly **7.5x** fewer iterations than no preconditioner and **5x** fewer than Jacobi at `anisotropy=50`.

## Summary

- **Jacobi** is cheap and always somewhat helpful, but blind to directional structure.
- **Line preconditioners** (tridiagonal for a non-periodic direction, circulant for a periodic one) do nothing for an isotropic problem, but scale far better than Jacobi as soon as one direction becomes stiffer than the other -- which is the common case for stretched grids and anisotropic media.
- Neither line preconditioner recovers the $O(1)$ conditioning a true 2D fast solver gets from a full 2D sine/cosine or Fourier transform, since each only removes one direction's coupling. Combining both directions (alternating direction implicit iteration, or a Woodbury-corrected exact circulant/tridiagonal factorisation) or going to a multigrid/AMG method (see [lineax#113](https://github.com/patrick-kidger/lineax/issues/113)) would be the natural next step for a solver that has to handle both regimes well.